In [1]:
#---------------------------
#라이브러리 임포트
#---------------------------

import os
import random

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, rdFMCS
from rdkit.DataStructs import TanimotoSimilarity

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA

# 시각화 스타일 설정
sns.set_style('whitegrid')

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
#-----------------
#파일 로드
#--------------
pub = pd.read_csv(" /summer_conference/open/Pubchem_ASK1.csv", low_memory=False)
pub = pub[pub['Activity_Value'] > 0].copy()
pub['pIC50'] = -np.log10(pub['Activity_Value'])
pub = pub[['SMILES', 'pIC50']].dropna()

chem = pd.read_csv(" /summer_conference/open/ChEMBL_ASK1(IC50).csv", sep=';')
chem = chem[['Smiles', 'pChEMBL Value']].dropna()
chem = chem.rename(columns={'Smiles': 'SMILES', 'pChEMBL Value': 'pIC50'})

df = pd.concat([pub, chem], ignore_index=True).drop_duplicates('SMILES').reset_index(drop=True)

In [3]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

def calc_all_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    try:
        # 기본 화학적 특징 계산
        feats = {
            'MolWt': Descriptors.MolWt(mol),
            'MolLogP': Descriptors.MolLogP(mol),
            'TPSA': Descriptors.TPSA(mol),
            'NumHDonors': Descriptors.NumHDonors(mol),
            'NumHAcceptors': Descriptors.NumHAcceptors(mol),
            'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
            'NumValenceElectrons': Descriptors.NumValenceElectrons(mol),
            'HeavyAtomCount': Descriptors.HeavyAtomCount(mol),
            'NumAliphaticRings': Descriptors.NumAliphaticRings(mol),
            'NumAromaticRings': rdMolDescriptors.CalcNumAromaticRings(mol),
            'NumRings': rdMolDescriptors.CalcNumRings(mol),
            'NumCharges': sum(a.GetFormalCharge() != 0 for a in mol.GetAtoms()),
            'NumCarbonAtoms': sum(a.GetSymbol() == 'C' for a in mol.GetAtoms())
        }
        
        # 주요 원소 개수
        atom_types = ['C', 'H', 'N', 'O', 'Cl', 'S', 'F', 'Br', 'I', 'B', 'Si']
        for atom in atom_types:
            feats[f'Num_{atom}'] = sum(a.GetSymbol() == atom for a in mol.GetAtoms())
        
        # 기능성 그룹 SMARTS 패턴
        fg_smarts = {
            'Amine_primary': '[NX3;H2]',
            'Amine_secondary': '[NX3;H1]',
            'Amine_tertiary': '[NX3;H0]',
            'Alcohol': '[OX2H]',
            'Aldehyde': '[CX3H1](=O)[#6]',
            'Ketone': '[#6][CX3](=O)[#6]',
            'Carboxylic_acid': '[CX3](=O)[OX2H1]',
            'Ester': '[CX3](=O)[OX2][#6]',
            'Nitrile': '[CX2]#N'
        }
        for name, patt in fg_smarts.items():
            smarts = Chem.MolFromSmarts(patt)
            feats[f'FG_{name}'] = len(mol.GetSubstructMatches(smarts))
        
        # Morgan Fingerprint (bit vector, 1024 bits, radius=2)
        fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
        fp_bits = {f'FP_{i}': int(bit) for i, bit in enumerate(fp.ToBitString())}
        
        # 모든 특징 합치기
        feats.update(fp_bits)
        
        return feats
    
    except Exception as e:
        print(f"Error processing SMILES {smiles}: {e}")
        return None


In [4]:
df['features'] = df['SMILES'].apply(calc_all_features)


[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerat

[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerator
[23:20:36] DEPRECATION WARNING: please use MorganGenerat

[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerator
[23:20:37] DEPRECATION WARNING: please use MorganGenerat

[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerat

[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerator
[23:20:38] DEPRECATION WARNING: please use MorganGenerat

[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerator
[23:20:39] DEPRECATION WARNING: please use MorganGenerat

[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerator
[23:20:40] DEPRECATION WARNING: please use MorganGenerat

In [5]:
# 딕셔너리를 컬럼으로 펼치기
features_df = pd.json_normalize(df['features'])

# 최종 데이터셋: pIC50 (목표값) + 모든 피처
final_df = pd.concat([df[['SMILES', 'pIC50']], features_df], axis=1)


In [6]:
print(final_df.head())

                                              SMILES     pIC50    MolWt  \
0  CC1=CC(=C(C=C1S(=O)(=O)N)C(=O)NC2=CC=CC(=N2)C3...  4.000000  430.490   
1  CCS(=O)(=O)NC1=CC(=C(C=C1)OC)C(=O)NC2=CC=CC(=N...  3.494850  444.517   
2  CNC(=O)C1CCN(CC1)C2=CN=C(C=C2N3C=C(N=C3)C4CC4)...  3.397940  558.672   
3  CC1=CC(=C(C=C1C(=O)N)C(=O)NC2=CC=CC(=N2)C3=NN=...  3.376751  394.435   
4  CC12C(C(CC(O1)N3C4=CC=CC=C4C5=C6C(=C7C8=CC=CC=...  3.161151  466.541   

   MolLogP    TPSA  NumHDonors  NumHAcceptors  NumRotatableBonds  \
0  2.13772  142.09           2              8                  6   
1  2.94350  128.10           2              8                  8   
2  3.40930  135.75           2             11                  8   
3  2.58922  125.02           2              7                  6   
4  4.35400   69.45           2              6                  2   

   NumValenceElectrons  HeavyAtomCount  ...  FP_1014  FP_1015  FP_1016  \
0                  158              30  ...        0        0     

In [7]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 테스트 데이터 예측
y_pred = model.predict(X_test)

# RMSE 계산
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# R^2 계산
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}") #작
print(f"R^2: {r2:.4f}") #클


NameError: name 'model' is not defined

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from sklearn.model_selection import train_test_split

# 화학적 특징 + Morgan Fingerprint 계산 함수
def calc_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # 기본 화학적 특징
    feats = {
        'MolWt': Descriptors.MolWt(mol),
        'MolLogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        'HeavyAtomCount': Descriptors.HeavyAtomCount(mol)
    }
    
    # Morgan Fingerprint (1024 bits, radius=2)
    fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    fp_bits = {f'FP_{i}': int(bit) for i, bit in enumerate(fp.ToBitString())}
    
    feats.update(fp_bits)
    return feats

# --- 1. PubChem 데이터 전처리 ---
pub = pd.read_csv(" /summer_conference/open/Pubchem_ASK1.csv", low_memory=False)

# IC50이고, "=" 관계인 데이터만 추출
pub_filtered = pub[
    (pub['Activity_Type'] == 'IC50') & 
    (pub['Activity_Qualifier'] == '=' ) & 
    (pub['Activity_Value'] > 0)  # 양수인 값만
].copy()

# pIC50 계산 (-log10(Activity_Value))
pub_filtered['pIC50'] = -np.log10(pub_filtered['Activity_Value'])

pub_filtered = pub_filtered[['SMILES', 'pIC50']].dropna()

# --- 2. ChEMBL 데이터 전처리 ---
chem = pd.read_csv(" /summer_conference/open/ChEMBL_ASK1(IC50).csv", sep=';')

# IC50 데이터만 필터링
chem_filtered = chem[
    (chem['Standard Type'] == 'IC50') & 
    (chem['Standard Relation'] == '=')
].copy()

# pIC50 컬럼으로 변환 (이미 pChEMBL Value가 -log10 변환된 값임)
chem_filtered = chem_filtered.rename(columns={'Smiles': 'SMILES', 'pChEMBL Value': 'pIC50'})
chem_filtered = chem_filtered[['SMILES', 'pIC50']].dropna()

# --- 3. 데이터 합치기 및 중복 제거 ---
df = pd.concat([pub_filtered, chem_filtered], ignore_index=True)
df = df.drop_duplicates('SMILES').reset_index(drop=True)

# --- 4. SMILES로부터 피처 생성 ---
feature_list = []
y_list = []

for i, row in df.iterrows():
    feats = calc_features(row['SMILES'])
    if feats is not None:
        feature_list.append(feats)
        y_list.append(row['pIC50'])

features_df = pd.DataFrame(feature_list)
y = np.array(y_list)

# 결측치 처리 (필요시)
features_df = features_df.fillna(0)

# --- 5. 학습/검증 데이터 분할 ---
X_train, X_test, y_train, y_test = train_test_split(
    features_df, y, test_size=0.2, random_state=42
)

print(f"Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

# 이제 X_train, y_train으로 모델 학습 가능


In [ ]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# ----------------------------
# Optuna objective function
# ----------------------------
def objective(trial):
    # 하이퍼파라미터 샘플링
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 5, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['auto', 'sqrt', 'log2']),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'random_state': 42,
        'n_jobs': -1
    }

    # 모델 생성 및 학습
    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    # 검증 예측
    y_pred = model.predict(X_valid)

    # 평가 지표: RMSE
    rmse = mean_squared_error(y_valid, y_pred, squared=False)
    return rmse  # Optuna는 값을 최소화함


In [ ]:
import pandas as pd
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# (1) X, y 데이터가 numpy라면 DataFrame/Series로 변환
X = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
y = pd.Series(y, name='target')

# (2) NaN 제거 (y 기준)
valid_idx = y.notna()
X = X.loc[valid_idx]
y = y.loc[valid_idx]

# (3) 데이터 분할
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# (4) Optuna 최적화 함수 정의
def objective(trial):
    # 하이퍼파라미터 샘플링
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', [ 'sqrt', 'log2']),
        'random_state': 42
    }

    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    # 예측 및 평가
    y_pred = model.predict(X_valid)
    rmse = mean_squared_error(y_valid, y_pred, squared=False)  # RMSE
    return rmse

# (5) Optuna 스터디 생성 및 실행
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

# (6) 최적 파라미터 확인
print("Best trial:")
print("  Value (RMSE):", study.best_trial.value)
print("  Params:")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")


In [ ]:
best_params = study.best_trial.params
best_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
best_model.fit(X_train, y_train)

# 예측
y_pred = best_model.predict(X_valid)


In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 1. 모델 생성 (기본 하이퍼파라미터, random_state 고정)
model = RandomForestRegressor(random_state=42)

# 2. 학습
model.fit(X_train, y_train)

# 3. 예측
y_pred = model.predict(X_test)

# 4. 평가 지표 계산
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

def custom_score(y_true_pIC50, y_pred_pIC50):
    # 1. pIC50 -> IC50(nM) 변환
    y_true_IC50 = 10 ** (-y_true_pIC50) * 1e9  # nM 단위 변환 (M -> nM)
    y_pred_IC50 = 10 ** (-y_pred_pIC50) * 1e9

    # 2. Normalized RMSE 계산 (RMSE / 1000 으로 정규화, 단 min 1 제한)
    rmse = np.sqrt(mean_squared_error(y_true_IC50, y_pred_IC50))
    normalized_rmse = min(rmse / 1000, 1)

    # 3. pIC50 기준 R^2 계산
    r2 = r2_score(y_true_pIC50, y_pred_pIC50)

    # 4. 최종 점수 계산
    score = 0.4 * (1 - normalized_rmse) + 0.6 * r2

    return {
        'Normalized_RMSE': normalized_rmse,
        'R2_pIC50': r2,
        'Final_Score': score
    }

# 사용 예시
results = custom_score(y_test, y_pred)
print(f"Normalized RMSE (IC50 nM): {results['Normalized_RMSE']:.4f}")
print(f"R^2 (pIC50): {results['R2_pIC50']:.4f}")
print(f"Final Score: {results['Final_Score']:.4f}")



NameError: name 'X_train' is not defined

In [9]:
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error

xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

# 평가
print("XGBoost R^2:", r2_score(y_test, y_pred_xgb))
print("XGBoost RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_xgb)))


NameError: name 'X_train' is not defined

In [10]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(
    random_state=42,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8
)

lgb_model.fit(X_train, y_train)

y_pred_lgb = lgb_model.predict(X_test)

# 평가
print("LightGBM R^2:", r2_score(y_test, y_pred_lgb))
print("LightGBM RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lgb)))


NameError: name 'X_train' is not defined

In [11]:
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import numpy as np

# === 데이터 준비 ===
# 이미 X_train, X_test, y_train, y_test 준비되어 있다고 가정

# === CatBoost 모델 정의 및 학습 ===
cat_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    random_state=42,
    verbose=0  # 로그 숨김
)

cat_model.fit(X_train, y_train)

# === 예측 ===
y_pred_cat = cat_model.predict(X_test)

# === 평가 지표 계산 ===
def evaluate_model(y_true, y_pred):
    # A: Normalized RMSE (IC50 기준)
    # 실제 IC50이 필요한 경우 pIC50 = -log10(IC50[nM]) → IC50 = 10^(-pIC50)
    ic50_true = 10 ** (-y_true)
    ic50_pred = 10 ** (-y_pred)
    
    rmse = np.sqrt(mean_squared_error(ic50_true, ic50_pred))
    norm_rmse = rmse / np.mean(ic50_true)

    # B: R² on pIC50
    r2 = r2_score(y_true, y_pred)

    # Score = 0.4 x (1 - min(A, 1)) + 0.6 x B
    score = 0.4 * (1 - min(norm_rmse, 1)) + 0.6 * r2

    return norm_rmse, r2, score

A_cat, B_cat, final_score_cat = evaluate_model(y_test, y_pred_cat)

print(f"[CatBoost] Normalized RMSE (A): {A_cat:.4f}")
print(f"[CatBoost] R² (B): {B_cat:.4f}")
print(f"[CatBoost] Final Score: {final_score_cat:.4f}")


NameError: name 'X_train' is not defined

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, rdMolDescriptors
import pandas as pd
import numpy as np
import os

# 파일 정보
file_infos = {
    "ChEMBL": {
        "path": " /summer_conference/open/ChEMBL_ASK1(IC50)_clean.csv",
        "type_col": "Standard Type"
    },
    "CAS": {
        "path": " /summer_conference/open/CAS_clean.csv",
        "type_col": "Assay Parameter"
    },
    "PubChem": {
        "path": " /summer_conference/open/Pubchem_clean.csv",
        "type_col": "Activity_Type"
    }
}

# 데이터 불러오기 및 정리
dfs = []
for source, info in file_infos.items():
    df = pd.read_csv(info["path"], encoding='cp1252')
    type_col = info["type_col"]

    if 'pIC50' in df.columns:
        pic50_col = 'pIC50'
    elif 'pX Value' in df.columns:
        pic50_col = 'pX Value'
    elif 'Assay Parameter' in df.columns:
        pic50_col = 'Assay Parameter'
    else:
        raise ValueError(f"{source}: pIC50 컬럼이 없습니다.")

    df = df[df[type_col].str.upper() == 'IC50']
    df = df[['SMILES', pic50_col]].copy()
    df.rename(columns={pic50_col: 'pIC50'}, inplace=True)
    df['Source'] = source
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

# Mol 생성
def safe_mol(smiles):
    try:
        return Chem.MolFromSmiles(smiles)
    except:
        return None

combined_df['mol'] = combined_df['SMILES'].apply(safe_mol)
combined_df = combined_df.dropna(subset=['mol'])

# 간단 피처
def extract_features(mol):
    return {
        'Num_N': sum(atom.GetSymbol() == 'N' for atom in mol.GetAtoms()),
    }

extra_feats = pd.DataFrame([extract_features(mol) for mol in combined_df['mol']])
df_with_extra = pd.concat([combined_df.reset_index(drop=True), extra_feats], axis=1)

# Morgan Fingerprint
def mol_to_morgan_fp(mol, radius=2, n_bits=2048):
    if mol is None:
        return np.zeros(n_bits, dtype=int)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    AllChem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

fp_array = np.array([mol_to_morgan_fp(mol) for mol in df_with_extra['mol']])
fp_df = pd.DataFrame(fp_array, columns=[f'FP_{i}' for i in range(fp_array.shape[1])])

# 분자 특성 피처
def featurize_full(mol):
    feats = {
        'MolWt': Descriptors.MolWt(mol),
        'MolLogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        'NumValenceElectrons': Descriptors.NumValenceElectrons(mol),
        'HeavyAtomCount': Descriptors.HeavyAtomCount(mol),
        'NumAliphaticRings': Descriptors.NumAliphaticRings(mol),
        'NumAromaticRings': rdMolDescriptors.CalcNumAromaticRings(mol),
        'NumRings': rdMolDescriptors.CalcNumRings(mol),
        'NumCharges': sum(a.GetFormalCharge() != 0 for a in mol.GetAtoms()),
        'NumCarbonAtoms': sum(a.GetSymbol() == 'C' for a in mol.GetAtoms())
    }

    atom_types = ['C', 'H', 'N', 'O', 'Cl', 'S', 'F', 'Br', 'I', 'B', 'Si']
    for atom in atom_types:
        feats[f'Num_{atom}'] = sum(a.GetSymbol() == atom for a in mol.GetAtoms())

    fg_smarts = {
        'Amine_primary': '[NX3;H2]',
        'Amine_secondary': '[NX3;H1]',
        'Amine_tertiary': '[NX3;H0]',
        'Alcohol': '[OX2H]',
        'Aldehyde': '[CX3H1](=O)[#6]',
        'Ketone': '[#6][CX3](=O)[#6]',
        'Carboxylic_acid': '[CX3](=O)[OX2H1]',
        'Ester': '[CX3](=O)[OX2][#6]',
        'Nitrile': '[CX2]#N'
    }
    for name, patt in fg_smarts.items():
        smarts = Chem.MolFromSmarts(patt)
        feats[f'FG_{name}'] = len(mol.GetSubstructMatches(smarts))

    return feats

full_feats = [featurize_full(mol) for mol in df_with_extra['mol']]
full_feat_df = pd.DataFrame(full_feats)

# 병합 (TPSA 중복 제거)
X_all = pd.concat([
    df_with_extra[['SMILES', 'pIC50', 'mol', 'Source', 'Num_N']],
    full_feat_df.reset_index(drop=True),
    fp_df.reset_index(drop=True)
], axis=1)

# 평가 함수
def evaluate_custom_score(y_true, y_pred):
    ic50_true = 10 ** (-y_true)
    ic50_pred = 10 ** (-y_pred)
    rmse_ic50 = np.sqrt(mean_squared_error(ic50_true, ic50_pred))
    nrmse = rmse_ic50 / np.mean(ic50_true)
    A = nrmse
    B = r2_score(y_true, y_pred)
    score = 0.4 * (1 - min(A, 1)) + 0.6 * B
    print(f"🔍 Normalized RMSE (A): {A:.3f}")
    print(f"🔍 R² (B): {B:.3f}")
    print(f"✅ 최종 Score: {score:.3f}\n")
    return A, B, score

# 학습 및 평가
def train_and_evaluate(model_name, X, y):
    X = X.drop(columns=['SMILES', 'pIC50', 'mol', 'Source'], errors='ignore')
    valid_idx = y.notnull() & X.notnull().all(axis=1)
    X_train, X_test, y_train, y_test = train_test_split(
        X[valid_idx], y[valid_idx], test_size=0.2, random_state=42)

    if model_name == 'RandomForest':
        model = RandomForestRegressor(random_state=42)
    elif model_name == 'XGBoost':
        model = XGBRegressor(random_state=42, verbosity=0)
    elif model_name == 'LightGBM':
        model = LGBMRegressor(random_state=42)
    else:
        raise ValueError(f"모델 '{model_name}'은 지원되지 않습니다.")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"\n📊 [{model_name}] 평가 결과")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")
    print(f"R²: {r2_score(y_test, y_pred):.3f}")
    evaluate_custom_score(y_test, y_pred)
    return model
X_all = X_all.loc[:, ~X_all.columns.duplicated()]
print("1번 성공")
# 디폴트 모델 학습 실행
# rf_model = train_and_evaluate('RandomForest', X_all, X_all['pIC50'])
# xgb_model = train_and_evaluate('XGBoost', X_all, X_all['pIC50'])
# lgbm_model = train_and_evaluate('LightGBM', X_all, X_all['pIC50'])


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerator
[23:32:20] DEPRECATION WARNING: please use MorganGenerat

1번 성공


In [3]:
from sklearn.model_selection import GridSearchCV

def train_with_grid_search(model_name, X, y):
    X = X.drop(columns=['SMILES', 'pIC50', 'mol', 'Source'], errors='ignore')
    valid_idx = y.notnull() & X.notnull().all(axis=1)
    X_train, X_test, y_train, y_test = train_test_split(
        X[valid_idx], y[valid_idx], test_size=0.2, random_state=42)

    if model_name == 'RandomForest':
        model = RandomForestRegressor(random_state=42)
        param_grid = {
            'n_estimators': [100, 200],
            'max_depth': [None, 10, 20],
            'max_features': ['auto', 'sqrt'],
            'min_samples_split': [2, 5]
        }
    elif model_name == 'XGBoost':
        model = XGBRegressor(random_state=42, verbosity=0)
        param_grid = {
            'n_estimators': [100, 200],
            'max_depth': [3, 6],
            'learning_rate': [0.01, 0.1],
            'subsample': [0.8, 1]
        }
    elif model_name == 'LightGBM':
        model = LGBMRegressor(random_state=42)
        param_grid = {
            'n_estimators': [100, 200],
            'max_depth': [-1, 10],
            'learning_rate': [0.01, 0.1],
            'num_leaves': [31, 50]
        }
    else:
        raise ValueError(f"모델 '{model_name}'은 지원되지 않습니다.")

    grid_search = GridSearchCV(model, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    print(f"\n📊 [{model_name} - GridSearchCV] 최적 하이퍼파라미터: {grid_search.best_params_}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")
    print(f"R²: {r2_score(y_test, y_pred):.3f}")
    evaluate_custom_score(y_test, y_pred)

    return best_model
print("완료-1")


완료-1


In [4]:
print("hello world")

hello world


In [4]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

def train_with_random_search(model_name, X, y, n_iter=20):
    X = X.drop(columns=['SMILES', 'pIC50', 'mol', 'Source'], errors='ignore')
    valid_idx = y.notnull() & X.notnull().all(axis=1)
    X_train, X_test, y_train, y_test = train_test_split(
        X[valid_idx], y[valid_idx], test_size=0.2, random_state=42)

    if model_name == 'RandomForest':
        model = RandomForestRegressor(random_state=42)
        param_dist = {
            'n_estimators': randint(50, 300),
            'max_depth': randint(5, 30),
            'max_features': ['auto', 'sqrt', 'log2'],
            'min_samples_split': randint(2, 10)
        }
    elif model_name == 'XGBoost':
        model = XGBRegressor(random_state=42, verbosity=0)
        param_dist = {
            'n_estimators': randint(50, 300),
            'max_depth': randint(3, 10),
            'learning_rate': uniform(0.01, 0.3),
            'subsample': uniform(0.7, 0.3)
        }
    elif model_name == 'LightGBM':
        model = LGBMRegressor(random_state=42)
        param_dist = {
            'n_estimators': randint(50, 300),
            'max_depth': randint(-1, 30),  # -1은 무제한
            'learning_rate': uniform(0.01, 0.3),
            'num_leaves': randint(20, 60)
        }
    else:
        raise ValueError(f"모델 '{model_name}'은 지원되지 않습니다.")

    random_search = RandomizedSearchCV(model, param_distributions=param_dist, 
                                       n_iter=n_iter, cv=3, scoring='neg_mean_squared_error',
                                       n_jobs=-1, random_state=42)
    random_search.fit(X_train, y_train)

    best_model = random_search.best_estimator_
    y_pred = best_model.predict(X_test)

    print(f"\n📊 [{model_name} - RandomizedSearchCV] 최적 하이퍼파라미터: {random_search.best_params_}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")
    print(f"R²: {r2_score(y_test, y_pred):.3f}")
    evaluate_custom_score(y_test, y_pred)

    return best_model
print("완료-1")


완료-1


In [ ]:
# Grid Search 예시
print("시작 그리드")
best_xgb_grid = train_with_grid_search('XGBoost', X_all, X_all['pIC50'])
print("완료-그리드")



시작 그리드
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instruc

In [ ]:
# Random Search 예시
best_xgb_random = train_with_random_search('XGBoost', X_all, X_all['pIC50'], n_iter=30)

In [ ]:
# Grid Search 예시
best_rf_grid = train_with_grid_search('RandomForest', X_all, X_all['pIC50'])

# Random Search 예시
best_xgb_random = train_with_random_search('XGBoost', X_all, X_all['pIC50'], n_iter=30)
